In [1]:
import pandas as pd
import numpy as np
import bioframe as bf

### Overlapping Akita.V2 and AlphaGenome test windows

In [ ]:
# MOUSE

# akita_path = "/project2/fudenber_735/tensorflow_models/akita/v2/data/mm10/sequences.bed"
# alphagenome_path = f"/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/mouse_cell_types/alphagenome_safe_windows.tsv"

In [2]:
# HUMAN

akita_path = "/project2/fudenber_735/tensorflow_models/akita/v2/data/hg38/sequences.bed"
alphagenome_path = f"/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/human_cell_types/alphagenome_safe_windows.tsv"

In [3]:
# reading test data for akita V2
sequences_akita = pd.read_csv(akita_path, sep='\t', names=['chr','start','end','type'])

In [4]:
# reading test data for alpha genome
sequences_test_alpha = pd.read_csv(alphagenome_path, sep='\t')

In [5]:
sequences_test_alpha.rename(columns={'chrom': 'chr'}, inplace=True)

In [6]:
df_overlap = bf.overlap(
        sequences_akita, sequences_test_alpha, how="inner", suffixes=("_akita", "_alpha"), cols1=["chr", "start", "end"], cols2=["chr", "start", "end"],
    )

In [ ]:
df_overlap

In [ ]:
# Keep only those Akita windows that overlap AlphaGenome windows all from the same fold,
# and exclude Akita windows that overlap AlphaGenome windows from multiple different folds.

In [7]:
# For each Akita window, check how many *unique* AlphaGenome folds it overlaps
folds_per_akita = (
    df_overlap.groupby(['chr_akita', 'start_akita', 'end_akita', 'type_akita'])['type_alpha']
    .nunique()
    .reset_index(name='n_unique_alpha_folds')
)

In [8]:
# Keep only Akita windows that overlap AlphaGenome windows from exactly one fold
akita_keep = folds_per_akita.query("n_unique_alpha_folds == 1")[['chr_akita', 'start_akita', 'end_akita', 'type_akita']]

In [9]:
# Filter df_overlap accordingly
df_overlap_filtered = df_overlap.merge(
    akita_keep,
    on=['chr_akita', 'start_akita', 'end_akita', 'type_akita'],
    how='inner'
)

In [10]:
# finding the closest windows
df_overlap_filtered["akita_midpoint"] = df_overlap_filtered["start_akita"] + (0.5*(df_overlap_filtered["end_akita"] - df_overlap_filtered["start_akita"]))
df_overlap_filtered["alpha_midpoint"] = df_overlap_filtered["start_alpha"] + (0.5*(df_overlap_filtered["end_alpha"] - df_overlap_filtered["start_alpha"]))
df_overlap_filtered["midpoint_dist"] = np.abs(df_overlap_filtered["akita_midpoint"]-df_overlap_filtered["alpha_midpoint"])

In [11]:
# selecting akita windows with minimal distance from the alpha genome windows
df_sorted = df_overlap_filtered.sort_values(by=['chr_akita', 'start_akita', 'end_akita', 'midpoint_dist'], ascending=[True, True, True, True])
df_unique = df_overlap_filtered.groupby(['chr_akita', 'start_akita', 'end_akita']).first().reset_index()

In [12]:
df_unique

,chr_akita,start_akita,end_akita,type_akita,chr_alpha,start_alpha,end_alpha,type_alpha,akita_midpoint,alpha_midpoint,midpoint_dist
0,chr1,66426880,67737600,fold0,chr1,67510483,68559059,fold0,67082240.0,68034771.0,952531.0
1,chr1,66754560,68065280,fold0,chr1,67510483,68559059,fold0,67409920.0,68034771.0,624851.0
2,chr1,67082240,68392960,fold0,chr1,67510483,68559059,fold0,67737600.0,68034771.0,297171.0
3,chr1,67409920,68720640,fold0,chr1,67510483,68559059,fold0,68065280.0,68034771.0,30509.0
4,chr1,67737600,69048320,fold0,chr1,67756348,68804924,fold0,68392960.0,68280636.0,112324.0
...,...,...,...,...,...,...,...,...,...,...,...
2748,chrX,139493376,140804096,fold0,chrX,139533348,140581924,fold1,140148736.0,140057636.0,91100.0
2749,chrX,141701120,143011840,fold0,chrX,141746133,142794709,fold1,142356480.0,142270421.0,86059.0
2750,chrX,142028800,143339520,fold0,chrX,142041171,143089747,fold1,142684160.0,142565459.0,118701.0
2751,chrX,142356480,143667200,fold0,chrX,142385382,143433958,fold1,143011840.0,142909670.0,102170.0


In [13]:
# renaming columns that are gonna be saved
df_unique = df_unique.rename(columns={"chr_akita" : "chr",
                                     "start_akita" : "start",
                                     "end_akita" : "end"})

In [14]:
# Keep only selected columns
df_to_save = df_unique[['chr', 'start', 'end', 'type_alpha', 'type_akita']]

In [15]:
df_to_save

,chr,start,end,type_alpha,type_akita
0,chr1,66426880,67737600,fold0,fold0
1,chr1,66754560,68065280,fold0,fold0
2,chr1,67082240,68392960,fold0,fold0
3,chr1,67409920,68720640,fold0,fold0
4,chr1,67737600,69048320,fold0,fold0
...,...,...,...,...,...
2748,chrX,139493376,140804096,fold1,fold0
2749,chrX,141701120,143011840,fold1,fold0
2750,chrX,142028800,143339520,fold1,fold0
2751,chrX,142356480,143667200,fold1,fold0


In [16]:
# for human only
# ORCA's test is chromosome 9 and 10

df_orca_filtered = df_to_save[df_to_save['chr'].isin(['chr9', 'chr10'])].copy()

In [17]:
df_orca_filtered

,chr,start,end,type_alpha,type_akita
153,chr10,49152,1359872,fold1,fold6
154,chr10,376832,1687552,fold1,fold6
155,chr10,704512,2015232,fold1,fold6
156,chr10,1032192,2342912,fold1,fold6
157,chr10,1359872,2670592,fold1,fold6
...,...,...,...,...,...
2514,chr9,135045120,136355840,fold3,fold5
2515,chr9,135372800,136683520,fold3,fold5
2516,chr9,135700480,137011200,fold3,fold5
2517,chr9,136028160,137338880,fold3,fold5


In [14]:
# sampling 500 random windows
df_sampled = df_to_save.sample(n=500, random_state=42)

In [ ]:
df_sampled = df_sampled.reset_index(drop=True)

In [ ]:
df_sampled 

In [ ]:
# Save as TSV (tab-separated, no index)
# df_sampled.to_csv(f"/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/mouse_cell_types/alphagenome_akita_test_overlap_500windows.tsv", sep='\t', index=False)

In [18]:
df_orca_filtered.to_csv(f"/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/human_cell_types/alphagenome_akita_orca_test_overlap.tsv", sep='\t', index=False)